# Human3.6M Body Orientation Classifier

**Pipeline:**
1. Load 3D joint data from Human3.6M
2. Compute hip vector → yaw angle → orientation label (Front/Side/Back)
3. Crop RGB frames using joint bounding boxes
4. Build a balanced 15K-image dataset
5. Train MobileNetV3-Small on the dataset
6. Evaluate with confusion matrix + cross-subject test (S9)

---
### Data paths
```
/path/to/human3.6m/
├── S1/
│   ├── MyPoseFeatures/
│   │   └── D3_Positions/
│   │       ├── Walking.cdf        # 3D joint positions
│   │       ├── WalkingDog.cdf
│   │       ├── Standing.cdf
│   │       └── Directions.cdf
│   └── Videos/
│       ├── Walking.54138969.mp4   # RGB videos (one per camera)
│       └── ...
├── S5/ ... S6/ ... S7/ ... S8/
```
Set `H36M_ROOT` below to your actual data path.
If you don't have the data yet, set `USE_MOCK_DATA = True` to test the full pipeline.


## 0 — Configuration & Imports

In [ ]:
# ─── Install dependencies (run once) ─────────────────────────────────────────
# Uncomment if needed:
# !pip install spacepy torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# !pip install matplotlib seaborn scikit-learn tqdm opencv-python
# spacepy is needed to read .cdf files from Human3.6M

In [37]:
import os
import math
import json
import random
import shutil
import warnings
from pathlib import Path
from math import atan2, pi
from typing import Dict, List, Tuple, Optional

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.gridspec import GridSpec
import seaborn as sns
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.models import MobileNet_V3_Small_Weights

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# ─── USER CONFIG — edit these paths ──────────────────────────────────────────
H36M_ROOT     = Path("/path/to/human3.6m")   # ← set your data root here
OUTPUT_ROOT   = Path("./dataset")             # where cropped images are saved
MODEL_SAVE    = Path("orientation_classifier.pth")

USE_MOCK_DATA = True   # ← set False when real data is available

SUBJECTS_TRAIN = [1, 5, 6, 7, 8]
SUBJECT_TEST   = 9
ACTIONS        = ['Walking', 'WalkingDog', 'Standing', 'Directions']

IMAGES_PER_CLASS = 5000   # 5k front, 5k side, 5k back
IMG_SIZE         = 224
BATCH_SIZE       = 32
EPOCHS           = 20
LR               = 1e-3
DEVICE           = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLASS_NAMES = ['Front', 'Side', 'Back']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

# Human3.6M joint index map (17-joint skeleton)
JOINT_NAMES = {
    0:  'Hip_Center',
    1:  'Spine',
    2:  'Thorax',
    3:  'Left_Hip',
    4:  'Left_Knee',
    5:  'Left_Ankle',
    6:  'Right_Hip',
    7:  'Right_Knee',
    8:  'Right_Ankle',
    9:  'Left_Shoulder',
    10: 'Left_Elbow',
    11: 'Left_Wrist',
    12: 'Head',
    13: 'Right_Shoulder',
    14: 'Right_Elbow',
    15: 'Right_Wrist',
    16: 'Neck',
}
J_LEFT_HIP   = 3
J_RIGHT_HIP  = 6
J_SPINE      = 1
J_HEAD       = 12

print(f"Device: {DEVICE}")
print(f"Mock mode: {USE_MOCK_DATA}")

Device: cuda
Mock mode: True


## 1 — Core Label Logic (reusable / Flask-ready)

In [42]:
# Constants for H36M Joint Indices
J_LEFT_HIP = 3
J_RIGHT_HIP = 6

def compute_orientation_label(
    joints_3d: np.ndarray,
    left_hip_idx: int = 3,
    right_hip_idx: int = 6,
    min_hip_width_mm: float = 50.0,
) -> dict:
    """
    Compute body orientation from 3D joint positions.
    Yaw Angle logic:
    0°   = Front (Hips are perfectly horizontal)
    90°  = Side (Right hip is far behind Left)
    180° = Back (Hips are horizontal but flipped)
    -90° = Side (Left hip is far behind Right)
    """
    left_hip  = joints_3d[left_hip_idx]
    right_hip = joints_3d[right_hip_idx]

    hip_vector = right_hip - left_hip  
    
    width = right_hip[0] - left_hip[0]
    depth_diff = right_hip[2] - left_hip[2] 

    abs_width = abs(width)
    # Validity check based on X-axis presence
    if abs_width < min_hip_width_mm and abs(depth_diff) < min_hip_width_mm:
        return {
            'label': None, 'angle_deg': None,
            'confidence': abs_width, 'hip_vector': hip_vector, 'valid': False
        }

    # atan2(y, x) -> we use depth as y and width as x
    angle_deg = atan2(depth_diff, width) * (180.0 / pi)

    # Classification logic
    abs_angle = abs(angle_deg)
    
    # 0 degrees means the hips are perfectly horizontal (Front)
    # 180 degrees means they are horizontal but swapped (Back)
    if abs_angle <= 30.0:
        label = 'Front'
    elif abs_angle >= 150.0:
        label = 'Back'
    else:
        label = 'Side'

    return {
        'label':      label,
        'angle_deg':  round(angle_deg, 2),
        'confidence': round(abs_width, 1),
        'hip_vector': hip_vector,
        'valid':      True,
    }

# ── Updated Unit Test with Corrected Logic ────────────────────────────────────
def _test_label_logic():
    """Sanity-check the orientation formula."""
    N = 17
    joints = np.zeros((N, 3), dtype=np.float32)

    # FRONT: Hips perfectly horizontal (Z=0, X=300) -> 0 degrees
    joints[J_LEFT_HIP]  = [-150, 0, 0] 
    joints[J_RIGHT_HIP] = [ 150, 0, 0] 
    r = compute_orientation_label(joints)
    assert r['label'] == 'Front', f"Expected Front, got {r}"

    # SIDE: One hip behind the other (Z=200, X=100) -> ~63 degrees
    joints[J_LEFT_HIP]  = [0, 0, 0]
    joints[J_RIGHT_HIP] = [100, 0, 200] 
    r = compute_orientation_label(joints)
    assert r['label'] == 'Side', f"Expected Side, got {r}"

    # BACK: Hips horizontal but Left is on the Right (X is negative) -> 180 degrees
    joints[J_LEFT_HIP]  = [ 150, 0, 0]
    joints[J_RIGHT_HIP] = [-150, 0, 0] 
    r = compute_orientation_label(joints)
    assert r['label'] == 'Back', f"Expected Back, got {r}"

    print("✅ Label logic tests passed!")

# Run the test
_test_label_logic()

✅ Label logic tests passed!


## 2 — Data Loader (Human3.6M CDF files)

In [43]:
def load_h36m_3d_joints(subject: int, action: str, root: Path) -> Optional[np.ndarray]:
    """
    Load 3D joint positions from a Human3.6M .cdf file.

    Expected file path:
        <root>/S<subject>/MyPoseFeatures/D3_Positions/<action>.cdf

    Returns
    -------
    np.ndarray of shape (T, 32, 3)  — T frames, 32 joints, x/y/z in mm
    (H36M stores 32 joints; we index into the 17-joint subset we care about)
    OR None if the file is missing.
    """
    cdf_path = root / f"S{subject}" / "MyPoseFeatures" / "D3_Positions" / f"{action}.cdf"
    if not cdf_path.exists():
        # Try alternate capitalizations / suffixes used in some H36M releases
        for alt in [f"{action} 1.cdf", f"{action}_1.cdf", f"{action}.55011271.cdf"]:
            alt_path = cdf_path.parent / alt
            if alt_path.exists():
                cdf_path = alt_path
                break
        else:
            print(f"  [WARN] Missing: {cdf_path}")
            return None

    try:
        from spacepy import pycdf
        with pycdf.CDF(str(cdf_path)) as cdf:
            # H36M stores pose as flat array of shape (1, T*96) or (T, 96)
            key = list(cdf.keys())[0]
            data = np.array(cdf[key])          # (1, T*96) or (T, 96)
            data = data.squeeze()              # (T*96,) or (T, 96)
            if data.ndim == 1:
                T = len(data) // 96
                data = data.reshape(T, 32, 3)
            else:
                data = data.reshape(-1, 32, 3)
        return data.astype(np.float32)
    except Exception as e:
        print(f"  [ERROR] Could not read {cdf_path}: {e}")
        return None


def load_h36m_video_frame(subject: int, action: str, frame_idx: int,
                          root: Path, camera: str = '54138969') -> Optional[np.ndarray]:
    """
    Extract a single RGB frame from a Human3.6M video.

    Expected file path:
        <root>/S<subject>/Videos/<action>.<camera>.mp4

    Returns BGR numpy array (H, W, 3) or None.
    """
    video_path = root / f"S{subject}" / "Videos" / f"{action}.{camera}.mp4"
    if not video_path.exists():
        for ext in ['.avi', '.mp4', '.mov']:
            alt = video_path.with_suffix(ext)
            if alt.exists():
                video_path = alt
                break
        else:
            return None

    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    cap.release()
    return frame if ret else None


def project_joints_to_2d(joints_3d: np.ndarray,
                          img_w: int = 1000, img_h: int = 1000) -> np.ndarray:
    """
    Very simple orthographic projection of 3D joints → 2D pixel coords.
    For proper perspective projection you need the H36M camera intrinsics.

    This is a reasonable approximation for bounding-box purposes.
    joints_3d: (N, 3) in mm
    Returns: (N, 2) in pixels
    """
    # Scale mm to a rough pixel range (H36M root joint is ~0, range ±1000mm)
    scale = img_w / 2000.0
    x_px = (joints_3d[:, 0] + 1000) * scale
    y_px = (joints_3d[:, 1] + 1000) * scale   # y+ = down
    return np.stack([x_px, y_px], axis=1).astype(np.float32)


print("Data loader functions defined.")

Data loader functions defined.


## 3 — Mock Data Generator (for testing without real data)

In [45]:
def generate_mock_joints(n_frames: int = 500, label: str = 'Front') -> np.ndarray:
    """
    Generate synthetic 3D joint sequences that produce a specific orientation label.

    The hip geometry is set to match the target label; other joints get
    small random offsets around anatomically plausible positions.

    Returns: (n_frames, 32, 3) float32 array in mm-scale
    """
    joints = np.zeros((n_frames, 32, 3), dtype=np.float32)

    # Base skeleton (standing T-pose, rough mm offsets from root)
    base = np.zeros((32, 3), dtype=np.float32)
    base[0]  = [   0,    0,    0]  # Hip center
    base[1]  = [   0,  100,    0]  # Spine
    base[2]  = [   0,  350,    0]  # Thorax
    base[3]  = [-100,  -50,    0]  # Left Hip
    base[4]  = [-100, -300,    0]  # Left Knee
    base[5]  = [-100, -550,    0]  # Left Ankle
    base[6]  = [ 100,  -50,    0]  # Right Hip
    base[7]  = [ 100, -300,    0]  # Right Knee
    base[8]  = [ 100, -550,    0]  # Right Ankle
    base[9]  = [-200,  350,    0]  # Left Shoulder
    base[10] = [-300,  150,    0]  # Left Elbow
    base[11] = [-380,  -50,    0]  # Left Wrist
    base[12] = [   0,  500,    0]  # Head
    base[13] = [ 200,  350,    0]  # Right Shoulder
    base[14] = [ 300,  150,    0]  # Right Elbow
    base[15] = [ 380,  -50,    0]  # Right Wrist
    base[16] = [   0,  430,    0]  # Neck

    # Add random walking motion
    t = np.linspace(0, 4 * np.pi, n_frames)
    walk_x = np.sin(t) * 30   # lateral sway
    walk_y = np.abs(np.sin(2 * t)) * 20  # vertical bounce

    for f in range(n_frames):
        frame_joints = base.copy()

        # Orientation-specific hip geometry
        if label == 'Front':
            # Hips at same Z depth, wide X spread (angle ~0°)
            frame_joints[J_LEFT_HIP]  = [-100 + walk_x[f],  -50, 0  + np.random.randn() * 5]
            frame_joints[J_RIGHT_HIP] = [ 100 + walk_x[f],  -50, 0  + np.random.randn() * 5]
        elif label == 'Side':
            # One hip much further in Z (angle ~±90°)
            z_offset = 150 + np.random.randn() * 20
            sign = 1 if f % 2 == 0 else -1
            frame_joints[J_LEFT_HIP]  = [-30 + walk_x[f], -50,  sign * z_offset]
            frame_joints[J_RIGHT_HIP] = [ 30 + walk_x[f], -50, -sign * z_offset]
        else:  # Back
            # Hips narrow X, large opposite Z (angle ~±180°)
            frame_joints[J_LEFT_HIP]  = [ -20 + walk_x[f], -50,  200 + np.random.randn() * 5]
            frame_joints[J_RIGHT_HIP] = [  20 + walk_x[f], -50, -200 + np.random.randn() * 5]

        # Add small noise to all joints
        frame_joints += np.random.randn(32, 3) * 3

        # Apply walking motion to all joints
        frame_joints[:, 0] += walk_x[f] * 0.3
        frame_joints[:, 1] += walk_y[f]

        joints[f] = frame_joints

    return joints


def generate_mock_frame(h: int = 1000, w: int = 1000) -> np.ndarray:
    """Generate a realistic-looking synthetic image (gradient + noise)."""
    base = np.random.randint(60, 180, (h, w, 3), dtype=np.uint8)
    # Add a gradient to simulate indoor background
    gradient = np.linspace(0, 40, w, dtype=np.uint8)
    base[:, :, 0] += gradient
    return base


# Quick test
mock_j = generate_mock_joints(10, 'Front')
result  = compute_orientation_label(mock_j[0])
print(f"Mock 'Front' frame → angle={result['angle_deg']}°, label={result['label']}")

mock_j = generate_mock_joints(10, 'Back')
result  = compute_orientation_label(mock_j[0])
print(f"Mock 'Back'  frame → angle={result['angle_deg']}°, label={result['label']}")

Mock 'Front' frame → angle=-0.21°, label=Front
Mock 'Back'  frame → angle=-85.13°, label=Side


## 4 — Build Dataset (crops + labels)

In [46]:
def build_dataset(
    subjects: List[int],
    actions: List[str],
    output_root: Path,
    images_per_class: int = 5000,
    use_mock: bool = False,
    h36m_root: Optional[Path] = None,
    split_ratios: Tuple[float, float, float] = (0.70, 0.15, 0.15),
) -> Dict:
    """
    Iterate over subjects/actions, compute orientation labels, crop frames,
    and save to a train/val/test folder structure.

    Returns a metadata dict with per-class counts and file paths.
    """
    splits = ['train', 'val', 'test']
    for split in splits:
        for cls in CLASS_NAMES:
            (output_root / split / cls).mkdir(parents=True, exist_ok=True)

    # Collect all valid (image_crop, label, angle, subject, action, frame_idx)
    collected = {cls: [] for cls in CLASS_NAMES}
    metadata  = []

    if use_mock:
        print("⚙️  Building mock dataset...")
        n_per_label = images_per_class + 500  # extra before de-duplication
        for label in CLASS_NAMES:
            joints_seq = generate_mock_joints(n_per_label, label)
            for f_idx in range(n_per_label):
                info = compute_orientation_label(joints_seq[f_idx])
                if not info['valid'] or info['label'] != label:
                    continue

                img = generate_mock_frame()
                joints_2d = project_joints_to_2d(joints_seq[f_idx], img.shape[1], img.shape[0])
                x1, y1, x2, y2 = get_bbox_from_joints(
                    joints_seq[f_idx], img.shape[1], img.shape[0], joints_2d
                )
                crop = img[y1:y2, x1:x2]
                if crop.size == 0:
                    continue
                crop = cv2.resize(crop, (IMG_SIZE, IMG_SIZE))
                collected[label].append({
                    'crop':     crop,
                    'label':    label,
                    'angle':    info['angle_deg'],
                    'subject':  -1,
                    'action':   'mock',
                    'frame':    f_idx,
                    'joints_3d': joints_seq[f_idx],
                    'joints_2d': joints_2d,
                })
                if len(collected[label]) >= images_per_class:
                    break
    else:
        print("⚙️  Building real dataset from Human3.6M...")
        for subj in subjects:
            for action in actions:
                print(f"  S{subj} / {action}")
                joints_seq = load_h36m_3d_joints(subj, action, h36m_root)
                if joints_seq is None:
                    continue

                n_frames = len(joints_seq)
                # Sample frames sparsely (every 5th) to avoid near-duplicate images
                frame_indices = list(range(0, n_frames, 5))
                random.shuffle(frame_indices)

                for f_idx in tqdm(frame_indices, desc=f"S{subj}/{action}", leave=False):
                    # Early exit per class if we have enough
                    label_info = compute_orientation_label(joints_seq[f_idx])
                    if not label_info['valid']:
                        continue
                    label = label_info['label']
                    if len(collected[label]) >= images_per_class:
                        continue

                    frame = load_h36m_video_frame(subj, action, f_idx, h36m_root)
                    if frame is None:
                        continue

                    h_img, w_img = frame.shape[:2]
                    joints_2d = project_joints_to_2d(joints_seq[f_idx], w_img, h_img)
                    x1, y1, x2, y2 = get_bbox_from_joints(
                        joints_seq[f_idx], w_img, h_img, joints_2d
                    )
                    crop = frame[y1:y2, x1:x2]
                    if crop.size == 0:
                        continue
                    crop = cv2.resize(crop, (IMG_SIZE, IMG_SIZE))

                    collected[label].append({
                        'crop':      crop,
                        'label':     label,
                        'angle':     label_info['angle_deg'],
                        'subject':   subj,
                        'action':    action,
                        'frame':     f_idx,
                        'joints_3d': joints_seq[f_idx],
                        'joints_2d': joints_2d,
                    })

                if all(len(v) >= images_per_class for v in collected.values()):
                    break

    # ── Balance & split ──────────────────────────────────────────────────────
    min_count = min(len(v) for v in collected.values())
    print(f"\nRaw counts: { {k: len(v) for k,v in collected.items()} }")
    print(f"Balancing to {min_count} per class")

    saved_paths = {}
    train_r, val_r, test_r = split_ratios

    for cls, items in collected.items():
        random.shuffle(items)
        items = items[:min_count]

        n_train = int(len(items) * train_r)
        n_val   = int(len(items) * val_r)

        split_items = {
            'train': items[:n_train],
            'val':   items[n_train:n_train + n_val],
            'test':  items[n_train + n_val:],
        }

        for split, split_list in split_items.items():
            for i, item in enumerate(split_list):
                fname = f"{cls}_{split}_{i:05d}.jpg"
                fpath = output_root / split / cls / fname
                cv2.imwrite(str(fpath), item['crop'])

                meta = {
                    'path':    str(fpath),
                    'label':   cls,
                    'split':   split,
                    'angle':   item['angle'],
                    'subject': item['subject'],
                    'action':  item['action'],
                    'frame':   item['frame'],
                }
                metadata.append(meta)

        saved_paths[cls] = {
            s: [m['path'] for m in metadata if m['label'] == cls and m['split'] == s]
            for s in splits
        }

    # Save metadata JSON
    meta_path = output_root / 'metadata.json'
    with open(meta_path, 'w') as f:
        json.dump(metadata, f, indent=2)

    print("\n✅ Dataset saved:")
    for split in splits:
        counts = {cls: len([m for m in metadata if m['split'] == split and m['label'] == cls])
                  for cls in CLASS_NAMES}
        print(f"  {split:5s}: {counts}")

    return metadata


# Build the dataset
metadata = build_dataset(
    subjects      = SUBJECTS_TRAIN,
    actions       = ACTIONS,
    output_root   = OUTPUT_ROOT,
    images_per_class = IMAGES_PER_CLASS,
    use_mock      = USE_MOCK_DATA,
    h36m_root     = H36M_ROOT if not USE_MOCK_DATA else None,
)

⚙️  Building mock dataset...

Raw counts: {'Front': 5000, 'Side': 5000, 'Back': 0}
Balancing to 0 per class

✅ Dataset saved:
  train: {'Front': 0, 'Side': 0, 'Back': 0}
  val  : {'Front': 0, 'Side': 0, 'Back': 0}
  test : {'Front': 0, 'Side': 0, 'Back': 0}


## 5 — Visualize 10 Sample Frames

In [ ]:
# Skeleton connectivity for drawing
SKELETON_EDGES = [
    (0, 1), (1, 2), (2, 16), (16, 12),   # spine chain
    (2, 9),  (9, 10),  (10, 11),           # left arm
    (2, 13), (13, 14), (14, 15),           # right arm
    (0, 3),  (3, 4),   (4, 5),             # left leg
    (0, 6),  (6, 7),   (7, 8),             # right leg
]

EDGE_COLORS = {
    'Front': '#2196F3',   # blue
    'Side':  '#FF9800',   # orange
    'Back':  '#E91E63',   # pink
}


def visualize_samples(metadata: List[Dict], n: int = 10,
                      output_root: Path = OUTPUT_ROOT):
    """Show n sample frames with skeleton overlay, angle, and label."""
    # Pick balanced samples
    samples = []
    per_class = n // len(CLASS_NAMES)
    for cls in CLASS_NAMES:
        cls_meta = [m for m in metadata if m['label'] == cls and m['split'] == 'train']
        samples.extend(random.sample(cls_meta, min(per_class, len(cls_meta))))
    random.shuffle(samples)
    samples = samples[:n]

    fig = plt.figure(figsize=(20, 4 * math.ceil(n / 5)))
    gs  = GridSpec(math.ceil(n / 5), 5, figure=fig, hspace=0.4, wspace=0.3)

    for i, meta in enumerate(samples):
        ax = fig.add_subplot(gs[i // 5, i % 5])

        # Load crop image
        img = cv2.imread(meta['path'])
        if img is None:
            img = generate_mock_frame(IMG_SIZE, IMG_SIZE)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img_rgb)

        label  = meta['label']
        angle  = meta['angle']
        color  = EDGE_COLORS.get(label, 'white')

        # Title with label + angle
        ax.set_title(
            f"{label} | {angle:.1f}°\n"
            f"S{meta['subject']} / {meta['action']}",
            fontsize=9, color='black',
            bbox=dict(boxstyle='round', facecolor=color, alpha=0.3)
        )

        # Draw bounding box indicator
        for spine in ax.spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(3)

        ax.axis('off')

    plt.suptitle('Sample Crops — with Computed Orientation Angle', fontsize=14, y=1.02)
    plt.savefig(str(OUTPUT_ROOT / 'sample_visualizations.png'),
                bbox_inches='tight', dpi=120)
    plt.show()
    print(f"Saved to {OUTPUT_ROOT / 'sample_visualizations.png'}")


visualize_samples(metadata, n=10)

## 6 — Angle Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of angles per class
ax = axes[0]
for cls in CLASS_NAMES:
    angles = [m['angle'] for m in metadata if m['label'] == cls]
    ax.hist(angles, bins=36, alpha=0.6, label=cls, color=list(EDGE_COLORS.values())[CLASS_NAMES.index(cls)])
ax.axvline(-45,  color='gray', linestyle='--', alpha=0.7, label='Boundaries')
ax.axvline( 45,  color='gray', linestyle='--', alpha=0.7)
ax.axvline(-135, color='gray', linestyle='--', alpha=0.7)
ax.axvline( 135, color='gray', linestyle='--', alpha=0.7)
ax.set_xlabel('Yaw Angle (°)')
ax.set_ylabel('Frame Count')
ax.set_title('Angle Distribution by Class')
ax.legend()

# Class balance bar chart
ax = axes[1]
for split in ['train', 'val', 'test']:
    counts = [len([m for m in metadata if m['label'] == cls and m['split'] == split])
              for cls in CLASS_NAMES]
    x = np.arange(len(CLASS_NAMES))
    offset = {'train': -0.25, 'val': 0, 'test': 0.25}[split]
    ax.bar(x + offset, counts, width=0.25, label=split, alpha=0.8)
ax.set_xticks(range(len(CLASS_NAMES)))
ax.set_xticklabels(CLASS_NAMES)
ax.set_ylabel('Count')
ax.set_title('Class Balance per Split')
ax.legend()

plt.tight_layout()
plt.savefig(str(OUTPUT_ROOT / 'data_analysis.png'), dpi=120, bbox_inches='tight')
plt.show()

## 7 — PyTorch Dataset & DataLoaders

In [ ]:
class OrientationDataset(Dataset):
    """
    Loads cropped human images from the folder structure:
        <root>/<split>/<class>/*.jpg

    Side images get a coin-flip horizontal flip during training
    (both orientations are equally valid for 'Side').
    """

    def __init__(self, root: Path, split: str, transform=None):
        self.root      = root
        self.split     = split
        self.transform = transform
        self.samples   = []

        for cls in CLASS_NAMES:
            cls_dir = root / split / cls
            if not cls_dir.exists():
                print(f"  [WARN] {cls_dir} not found")
                continue
            for fpath in sorted(cls_dir.glob('*.jpg')):
                self.samples.append((str(fpath), CLASS_TO_IDX[cls]))

        print(f"{split:5s}: {len(self.samples):,} images")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label_idx = self.samples[idx]
        img = cv2.imread(path)
        if img is None:
            # Return a blank tensor if the file is missing
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        from PIL import Image
        img = Image.fromarray(img)

        if self.transform:
            img = self.transform(img)

        return img, label_idx


# ── Transforms ────────────────────────────────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(
        brightness=0.2, contrast=0.2, saturation=0.15, hue=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ── DataLoaders ───────────────────────────────────────────────────────────────
train_ds = OrientationDataset(OUTPUT_ROOT, 'train', train_transform)
val_ds   = OrientationDataset(OUTPUT_ROOT, 'val',   val_transform)
test_ds  = OrientationDataset(OUTPUT_ROOT, 'test',  val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"\nBatches — train: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}")

## 8 — Model: MobileNetV3-Small

In [ ]:
def build_mobilenetv3_small(num_classes: int = 3, dropout: float = 0.2) -> nn.Module:
    """
    MobileNetV3-Small pretrained on ImageNet.
    Only the classifier head is replaced; backbone is frozen for the first
    few epochs then gradually unfrozen.
    """
    model = models.mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)

    # Freeze backbone (features) initially
    for param in model.features.parameters():
        param.requires_grad = False

    # Replace classifier head
    in_features = model.classifier[3].in_features
    model.classifier[3] = nn.Linear(in_features, num_classes)

    return model


model = build_mobilenetv3_small(num_classes=len(CLASS_NAMES)).to(DEVICE)

# Count params
total  = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params:     {total:,}")
print(f"Trainable params: {trainable:,}  ({100*trainable/total:.1f}%)")

## 9 — Training

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3, verbose=True
)


def run_epoch(loader, model, criterion, optimizer=None, device=DEVICE):
    """Run one train or eval epoch. Returns (avg_loss, accuracy)."""
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, correct, total = 0.0, 0, 0

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)

            logits = model(imgs)
            loss   = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * len(labels)
            correct    += (logits.argmax(1) == labels).sum().item()
            total      += len(labels)

    return total_loss / total, correct / total


def unfreeze_backbone(model, epoch, unfreeze_epoch=5):
    """Gradually unfreeze backbone after a warm-up period."""
    if epoch == unfreeze_epoch:
        print(f"  → Unfreezing backbone at epoch {epoch}")
        for param in model.features.parameters():
            param.requires_grad = True
        # Reinitialize optimizer to include new params
        return optim.Adam(model.parameters(), lr=LR / 5)
    return None


# ── Training loop ─────────────────────────────────────────────────────────────
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
patience_counter = 0
EARLY_STOP_PATIENCE = 7

for epoch in range(1, EPOCHS + 1):

    # Optionally unfreeze backbone mid-training
    new_opt = unfreeze_backbone(model, epoch)
    if new_opt is not None:
        optimizer = new_opt
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=0.5, patience=3, verbose=True
        )

    tr_loss, tr_acc = run_epoch(train_loader, model, criterion, optimizer)
    vl_loss, vl_acc = run_epoch(val_loader,   model, criterion)

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(vl_loss)
    history['val_acc'].append(vl_acc)

    scheduler.step(vl_acc)

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), MODEL_SAVE)
        patience_counter = 0
        flag = ' ✅ saved'
    else:
        patience_counter += 1
        flag = f' (patience {patience_counter}/{EARLY_STOP_PATIENCE})'

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train loss={tr_loss:.4f} acc={tr_acc:.4f} | "
        f"Val loss={vl_loss:.4f} acc={vl_acc:.4f}{flag}"
    )

    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"Early stopping at epoch {epoch}")
        break

print(f"\nBest val accuracy: {best_val_acc:.4f}")

## 10 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_run = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_run, history['train_loss'], label='Train', marker='o', ms=4)
axes[0].plot(epochs_run, history['val_loss'],   label='Val',   marker='s', ms=4)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('Loss Curve'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_run, history['train_acc'], label='Train', marker='o', ms=4)
axes[1].plot(epochs_run, history['val_acc'],   label='Val',   marker='s', ms=4)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy Curve'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig(str(OUTPUT_ROOT / 'training_curves.png'), dpi=120, bbox_inches='tight')
plt.show()

## 11 — Evaluation: Confusion Matrix on Test Set

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load(MODEL_SAVE, map_location=DEVICE))
model.eval()


def evaluate_loader(loader, model, device=DEVICE):
    """Returns (y_true, y_pred, confidences) for the whole loader."""
    y_true, y_pred, confs = [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            logits = model(imgs)
            probs  = torch.softmax(logits, dim=1)
            preds  = probs.argmax(dim=1).cpu().numpy()
            conf   = probs.max(dim=1).values.cpu().numpy()
            y_true.extend(labels.numpy())
            y_pred.extend(preds)
            confs.extend(conf)
    return np.array(y_true), np.array(y_pred), np.array(confs)


y_true, y_pred, confs = evaluate_loader(test_loader, model)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title('Confusion Matrix (Counts)')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set_title('Confusion Matrix (Normalized)')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')

plt.suptitle('Test Set — Confusion Matrix', fontsize=14)
plt.tight_layout()
plt.savefig(str(OUTPUT_ROOT / 'confusion_matrix.png'), dpi=120, bbox_inches='tight')
plt.show()

# Classification report
print("\nTest Set Classification Report")
print("=" * 50)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# Overall test accuracy
test_acc = (y_true == y_pred).mean()
print(f"Overall test accuracy: {test_acc:.4f}")

## 12 — Misclassified Example Analysis

In [ ]:
def show_misclassified(model, dataset: OrientationDataset,
                       metadata_list: List[Dict],
                       n: int = 12, device=DEVICE):
    """
    Find and display misclassified examples.
    For each, show the image, the computed yaw angle, true label, and predicted label.
    """
    model.eval()
    wrong = []

    meta_test = {m['path']: m for m in metadata_list if m['split'] == 'test'}

    with torch.no_grad():
        for img_path, true_idx in dataset.samples:
            from PIL import Image as PILImage
            img_pil = PILImage.open(img_path).convert('RGB')
            x = val_transform(img_pil).unsqueeze(0).to(device)
            logits = model(x)
            pred_idx = logits.argmax(1).item()

            if pred_idx != true_idx:
                meta = meta_test.get(img_path, {})
                wrong.append({
                    'path':       img_path,
                    'true_label': CLASS_NAMES[true_idx],
                    'pred_label': CLASS_NAMES[pred_idx],
                    'angle':      meta.get('angle', 'N/A'),
                    'subject':    meta.get('subject', '?'),
                    'action':     meta.get('action', '?'),
                    'confidence': torch.softmax(logits, 1).max().item(),
                })
            if len(wrong) >= n * 4:
                break

    if not wrong:
        print("No misclassified examples found — perfect test score! 🎉")
        return

    print(f"Found {len(wrong)} misclassified examples. Showing {min(n, len(wrong))}.")

    cols = min(4, n)
    rows = math.ceil(min(n, len(wrong)) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    axes = np.array(axes).flatten()

    for i, item in enumerate(wrong[:n]):
        ax = axes[i]
        img = cv2.imread(item['path'])
        if img is not None:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(
            f"True: {item['true_label']}\n"
            f"Pred: {item['pred_label']} ({item['confidence']:.2f})\n"
            f"Angle: {item['angle']}°",
            fontsize=9,
            color='red'
        )
        # Red border
        for spine in ax.spines.values():
            spine.set_edgecolor('red'); spine.set_linewidth(2)
        ax.axis('off')

    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.suptitle('Misclassified Samples — True vs Predicted', fontsize=13)
    plt.tight_layout()
    plt.savefig(str(OUTPUT_ROOT / 'misclassified.png'), dpi=120, bbox_inches='tight')
    plt.show()

    # Print summary table
    print("\nMisclassified Details:")
    print(f"{'True':8} {'Pred':8} {'Angle':>8} {'Conf':>6} {'Subject':>8} Action")
    print("-" * 55)
    for item in wrong[:20]:
        print(
            f"{item['true_label']:8} {item['pred_label']:8} "
            f"{str(item['angle']):>8} {item['confidence']:>6.3f} "
            f"S{item['subject']:>7} {item['action']}"
        )


show_misclassified(model, test_ds, metadata)

## 13 — Cross-Subject Generalization Test (Subject 9)

In [ ]:
def evaluate_subject(
    subject: int,
    actions: List[str],
    model: nn.Module,
    h36m_root: Path,
    use_mock: bool = False,
    device=DEVICE,
):
    """
    Run the full pipeline on a held-out subject and report accuracy.
    Also shows examples where the geometric label and the model disagree.
    """
    model.eval()
    results = []

    if use_mock:
        print(f"⚙️  Generating mock data for S{subject}...")
        for label in CLASS_NAMES:
            joints_seq = generate_mock_joints(200, label)
            for f_idx in range(200):
                info = compute_orientation_label(joints_seq[f_idx])
                if not info['valid']:
                    continue

                img = generate_mock_frame()
                joints_2d = project_joints_to_2d(joints_seq[f_idx], img.shape[1], img.shape[0])
                x1, y1, x2, y2 = get_bbox_from_joints(
                    joints_seq[f_idx], img.shape[1], img.shape[0], joints_2d
                )
                crop = img[y1:y2, x1:x2]
                if crop.size == 0:
                    continue
                crop = cv2.resize(crop, (IMG_SIZE, IMG_SIZE))

                from PIL import Image as PILImage
                crop_pil = PILImage.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
                x = val_transform(crop_pil).unsqueeze(0).to(device)

                with torch.no_grad():
                    logits = model(x)
                    pred_idx  = logits.argmax(1).item()
                    pred_conf = torch.softmax(logits, 1).max().item()

                results.append({
                    'true_label': info['label'],
                    'pred_label': CLASS_NAMES[pred_idx],
                    'angle':      info['angle_deg'],
                    'confidence': pred_conf,
                    'correct':    info['label'] == CLASS_NAMES[pred_idx],
                })
    else:
        for action in actions:
            joints_seq = load_h36m_3d_joints(subject, action, h36m_root)
            if joints_seq is None:
                continue
            for f_idx in tqdm(range(0, len(joints_seq), 10),
                              desc=f"S{subject}/{action}"):
                info = compute_orientation_label(joints_seq[f_idx])
                if not info['valid']:
                    continue

                frame = load_h36m_video_frame(subject, action, f_idx, h36m_root)
                if frame is None:
                    continue

                h_img, w_img = frame.shape[:2]
                joints_2d = project_joints_to_2d(joints_seq[f_idx], w_img, h_img)
                x1, y1, x2, y2 = get_bbox_from_joints(
                    joints_seq[f_idx], w_img, h_img, joints_2d
                )
                crop = frame[y1:y2, x1:x2]
                if crop.size == 0:
                    continue
                crop = cv2.resize(crop, (IMG_SIZE, IMG_SIZE))

                from PIL import Image as PILImage
                crop_pil = PILImage.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
                x = val_transform(crop_pil).unsqueeze(0).to(device)

                with torch.no_grad():
                    logits = model(x)
                    pred_idx  = logits.argmax(1).item()
                    pred_conf = torch.softmax(logits, 1).max().item()

                results.append({
                    'true_label': info['label'],
                    'pred_label': CLASS_NAMES[pred_idx],
                    'angle':      info['angle_deg'],
                    'confidence': pred_conf,
                    'correct':    info['label'] == CLASS_NAMES[pred_idx],
                })

    if not results:
        print(f"No results for S{subject}")
        return

    acc = np.mean([r['correct'] for r in results])

    y_t = [CLASS_TO_IDX[r['true_label']] for r in results]
    y_p = [CLASS_TO_IDX[r['pred_label']] for r in results]

    print(f"\n{'='*55}")
    print(f"Cross-Subject Test — Subject {subject}")
    print(f"{'='*55}")
    print(f"Total frames evaluated: {len(results):,}")
    print(f"Accuracy:               {acc:.4f}")
    print()
    print(classification_report(y_t, y_p, target_names=CLASS_NAMES))

    # Confusion matrix
    cm = confusion_matrix(y_t, y_p)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f'Subject {subject} — Cross-Subject Generalization')
    plt.xlabel('Predicted'); plt.ylabel('True')
    plt.tight_layout()
    plt.savefig(str(OUTPUT_ROOT / f'cross_subject_S{subject}.png'), dpi=120)
    plt.show()

    # Disagreement examples
    wrong = [r for r in results if not r['correct']]
    print(f"\nMisclassified: {len(wrong)} ({100*len(wrong)/len(results):.1f}%)")
    if wrong:
        print(f"{'True':8} {'Pred':8} {'Angle':>8} {'Conf':>6}")
        print("-" * 36)
        for r in wrong[:15]:
            print(
                f"{r['true_label']:8} {r['pred_label']:8} "
                f"{r['angle']:>8.1f} {r['confidence']:>6.3f}"
            )

    return results


cross_subject_results = evaluate_subject(
    subject   = SUBJECT_TEST,
    actions   = ACTIONS,
    model     = model,
    h36m_root = H36M_ROOT,
    use_mock  = USE_MOCK_DATA,
)

## 14 — Export: Flask-Ready Label Function

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Copy this entire cell into your Flask app (orientation_utils.py)
# ─────────────────────────────────────────────────────────────────────────────

FLASK_EXPORT_CODE = '''
# orientation_utils.py
# Generated by Human3.6M orientation pipeline

import math
import numpy as np
import torch
import torchvision.transforms as transforms
from torchvision.models import mobilenet_v3_small
import cv2
from PIL import Image

CLASS_NAMES   = ["Front", "Side", "Back"]
CLASS_TO_IDX  = {c: i for i, c in enumerate(CLASS_NAMES)}
IMG_SIZE      = 224
J_LEFT_HIP    = 3
J_RIGHT_HIP   = 6

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

preprocess = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def compute_orientation_label(joints_3d, left_hip_idx=3, right_hip_idx=6,
                               min_hip_width_mm=50.0):
    """
    Compute orientation from 3D joints (mm).
    Returns dict with keys: label, angle_deg, confidence, valid
    """
    left_hip  = joints_3d[left_hip_idx]
    right_hip = joints_3d[right_hip_idx]
    depth_diff = right_hip[2] - left_hip[2]
    width = abs(right_hip[0] - left_hip[0])

    if width < min_hip_width_mm:
        return {"label": None, "angle_deg": None, "confidence": width, "valid": False}

    angle_deg = math.atan2(depth_diff, width) * (180.0 / math.pi)
    abs_a = abs(angle_deg)

    if abs_a <= 45.0:
        label = "Front"
    elif abs_a <= 135.0:
        label = "Side"
    else:
        label = "Back"

    return {"label": label, "angle_deg": round(angle_deg, 2),
            "confidence": round(width, 1), "valid": True}


def load_model(checkpoint_path: str, device="cpu"):
    """Load the trained MobileNetV3-Small orientation classifier."""
    model = mobilenet_v3_small(weights=None)
    in_features = model.classifier[3].in_features
    import torch.nn as nn
    model.classifier[3] = nn.Linear(in_features, len(CLASS_NAMES))
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()
    return model.to(device)


def predict_from_crop(model, crop_bgr: np.ndarray, device="cpu") -> dict:
    """
    Run the classifier on a BGR crop (numpy array).
    Returns dict: {label, confidence, probabilities}
    """
    img = Image.fromarray(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
    x = preprocess(img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(x)
        probs  = torch.softmax(logits, 1).squeeze().cpu().numpy()

    pred_idx = int(probs.argmax())
    return {
        "label":         CLASS_NAMES[pred_idx],
        "confidence":    float(probs[pred_idx]),
        "probabilities": {c: float(p) for c, p in zip(CLASS_NAMES, probs)},
    }


# ── Flask usage example ────────────────────────────────────────────────────
# from orientation_utils import load_model, predict_from_crop, compute_orientation_label
#
# model = load_model("orientation_classifier.pth")
#
# @app.route("/predict", methods=["POST"])
# def predict():
#     file   = request.files["image"]
#     img    = cv2.imdecode(np.frombuffer(file.read(), np.uint8), cv2.IMREAD_COLOR)
#     result = predict_from_crop(model, img)
#     return jsonify(result)
'''

# Save the Flask utility file
flask_path = Path('orientation_utils.py')
# Strip the triple-quoted wrapper
code_lines = FLASK_EXPORT_CODE.strip().splitlines()
# Remove the surrounding quotes line if present
clean_code = '\n'.join(code_lines)
flask_path.write_text(clean_code)
print(f"✅ Flask utility saved to: {flask_path.resolve()}")
print(f"   Functions exported: compute_orientation_label, load_model, predict_from_crop")

## 15 — Summary

In [ ]:
print("=" * 60)
print("PIPELINE SUMMARY")
print("=" * 60)

total_samples = len(metadata)
split_counts = {s: len([m for m in metadata if m['split'] == s]) for s in ['train', 'val', 'test']}
class_counts = {c: len([m for m in metadata if m['label'] == c]) for c in CLASS_NAMES}

print(f"\nDataset")
print(f"  Total images  : {total_samples:,}")
print(f"  Train/Val/Test: {split_counts['train']:,} / {split_counts['val']:,} / {split_counts['test']:,}")
print(f"  Class balance : { {k: v for k,v in class_counts.items()} }")

print(f"\nModel")
print(f"  Architecture : MobileNetV3-Small (pretrained ImageNet)")
print(f"  Saved to     : {MODEL_SAVE.resolve()}")
print(f"  Best val acc : {best_val_acc:.4f}")

test_acc_final = (y_true == y_pred).mean()
print(f"\nEvaluation")
print(f"  Test accuracy (in-distribution) : {test_acc_final:.4f}")
if cross_subject_results:
    cs_acc = np.mean([r['correct'] for r in cross_subject_results])
    print(f"  Cross-subject accuracy (S{SUBJECT_TEST})      : {cs_acc:.4f}")

print(f"\nExported files")
print(f"  orientation_utils.py  — Flask-ready inference utilities")
print(f"  {OUTPUT_ROOT}/metadata.json — full frame-level labels")
print(f"  {OUTPUT_ROOT}/sample_visualizations.png")
print(f"  {OUTPUT_ROOT}/confusion_matrix.png")
print(f"  {OUTPUT_ROOT}/training_curves.png")

print("\n" + "=" * 60)